# Synthetic Data — Transformer Diffusion Model
### Empirical memorization & generation quality on hypersphere manifold data (DiT + Flow Matching)

**Purpose:** Test whether theoretical insights about optimal latent dimensionality hold when the
intrinsic dimension is *known exactly* via controlled synthetic data on a low-dimensional manifold.

| Quantity | Symbol | Definition |
|---|---|---|
| Generation onset | τ_gen | Training step when AFD drops below threshold |
| Memorization onset | τ_mem | Training step when memo ratio exceeds threshold |
| Optimal latent dim | d* | Value of d_latent minimising AFD_test |
| Intrinsic dim | d_int | Known exactly: n−1 for S^{n−1} (+ FLIPD estimate) |

**Hypothesis:** `d* ≈ d_int`; larger `d_latent` leads to earlier `τ_mem`.  
**Data:** Uniform samples on S^{n−1} (unit hypersphere in R^n), embedded in R^512 ambient space.  
d_intrinsic swept from 4 to 256 (sphere_n ∈ {5, 9, 17, 33, 65, 129, 257}).  
A Conv-VAE reshapes the ambient vectors into pseudo-images and compresses to latent tensors for DiT;
ambient-space Fréchet distance (AFD) replaces Inception FID.  Generalises Bonnaire's circle (S^1) to arbitrary dimension.

---
## 0 · Environment setup

In [ ]:
import os, sys, subprocess

os.environ.update({
    'TF_CPP_MIN_LOG_LEVEL':         '3',
    'TF_ENABLE_ONEDNN_OPTS':         '0',
    'CUDA_MODULE_LOADING':           'LAZY',
    'XLA_PYTHON_CLIENT_PREALLOCATE': 'false',
    'GRPC_VERBOSITY':                'ERROR',
    'GLOG_minloglevel':              '3',
})
import warnings, logging
warnings.filterwarnings('ignore', category=UserWarning)
for _log in ('tensorflow', 'jax', 'absl'): logging.getLogger(_log).setLevel(logging.ERROR)

IN_KAGGLE = os.path.exists('/kaggle/working')
IN_COLAB  = 'google.colab' in sys.modules
IN_VERTEX = os.environ.get('VERTEX_AI_USER_MANAGED_NOTEBOOKS') == 'true'
IN_GCP    = IN_COLAB or IN_VERTEX or os.environ.get('CLOUD_ML_PROJECT_ID') is not None
ENV       = 'kaggle' if IN_KAGGLE else ('gcp' if IN_GCP else 'local')
print(f'Environment: {ENV.upper()}')

if IN_KAGGLE:
    OUT_DIR = '/kaggle/working/synthetic'
elif IN_VERTEX:
    GCS_BUCKET = os.environ.get('AIP_MODEL_DIR', '')
    OUT_DIR = GCS_BUCKET if GCS_BUCKET else '/home/jupyter/synthetic'
else:
    OUT_DIR = './synthetic'

CKPT_DIR = os.path.join(OUT_DIR, 'checkpoints')
os.makedirs(OUT_DIR,   exist_ok=True)
os.makedirs(CKPT_DIR,  exist_ok=True)
print(f'Output → {OUT_DIR}')

def pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

for pkg, imp in [('scipy','scipy'), ('einops','einops')]:
    try:
        __import__(imp)
    except ImportError:
        pip(pkg)

print('Deps ready.')

In [ ]:
import torch, numpy as np, random

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__}  |  device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'  GPU : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
    # Enable TF32 on Ampere GPUs for ~2x throughput
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32       = True

---
## 1 · Configuration

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  MULTI-RUN SETUP: Change RUN_ID for each Kaggle session.
#  Each run sweeps different sphere dimensions (d_intrinsic = sphere_n − 1).
#  d_latent = [16, 32, 64, 128, 256] straddles d_intrinsic across runs 1-3.
#  d_ambient = 512: large ambient space so high-d_int manifolds are embedded.
#  After all runs, the merge cell combines results and plots the dashboard.
# ══════════════════════════════════════════════════════════════════════════════

RUN_ID = 1  # ← change to 2 or 3 for subsequent Kaggle sessions

RUN_SCHEDULE = {
    1: [5, 9, 17],          # sphere_n → d_intrinsic = 4, 8, 16
    2: [33, 65],            # d_intrinsic = 32, 64
    3: [129, 257],          # d_intrinsic = 128, 256
}

INPUT_CKPT_DIR = None

CFG = dict(
    # ── Manifold data ─────────────────────────────────────────────────────────
    dataset        = 'hypersphere',
    sphere_n_grid  = RUN_SCHEDULE[RUN_ID],   # sphere dims for this run
    d_ambient      = 512,        # ambient embedding dimension (≥ max sphere_n)
    n_classes      = 0,          # unconditional generation

    # ── Latent space sweep ────────────────────────────────────────────────────
    # Each entry is the number of latent channels c;
    # latent shape = (c, latent_spatial, latent_spatial)
    # → d_latent = c × latent_spatial²
    latent_channels = [1, 2, 4, 8, 16],
    latent_spatial  = 4,    # h=w=4 → d_latent = c × 16

    # ── DiT architecture ──────────────────────────────────────────────────────
    dit_variant    = 'S',   # 'S' | 'B' | 'L' | 'XL'
    patch_size     = 2,     # p×p patchification of the latent spatial grid

    # ── Flow matching ─────────────────────────────────────────────────────────
    objective      = 'flow_matching',
    fm_sample_steps = 50,        # Euler ODE steps for sampling

    # ── Training ──────────────────────────────────────────────────────────────
    total_steps    = 80_000,     # gradient steps per (d_latent, d_int) run
    batch_size     = 128,
    lr             = 1e-4,
    ema_decay      = 0.9999,
    grad_clip      = 1.0,
    mixed_prec     = True,       # AMP fp16/bf16

    # ── Dataset ───────────────────────────────────────────────────────────────
    n_train        = 10_000,     # fixed training set size
    n_test         = 10_000,     # fixed test set size

    # ── VAE ───────────────────────────────────────────────────────────────────
    # ConvVAE reshapes R^{d_ambient} → (in_c, 2S, 2S) pseudo-image, where
    # S = latent_spatial.  Requires d_ambient % (2*latent_spatial)² == 0.
    vae_epochs     = 30,         # epochs for ConvVAE pre-training
    vae_kl_weight  = 1e-6,       # β weight for KL term (per-sample sum KL)

    # ── Evaluation ────────────────────────────────────────────────────────────
    loss_every     = 500,        # record training loss every N steps
    eval_every     = 5_000,      # measure metrics every N steps
    n_fid_samples  = 5_000,      # samples for FD at each checkpoint
    fid_threshold  = 50.0,       # τ_gen: AFD below this → generation has 'started'
    memo_threshold = 0.075,      # τ_mem: NN memo rate above this → memorisation
    memo_nn_thresh = 0.5,        # d1/d2 ratio threshold per sample (Carlini/Gu)

    # ── FLIPD ─────────────────────────────────────────────────────────────────
    flipd_n_probes = 15,          # Hutchinson probes for velocity Jacobian trace
    flipd_t_vals   = [0.005, 0.01, 0.05, 0.1],  # noise levels t ∈ [0,1]
    flipd_primary_t = 0.01,      # t value used for d_intrinsic estimate

    # ── Misc ──────────────────────────────────────────────────────────────────
    seed           = 42,
    num_workers    = 0,          # synthetic data is in-memory
    save_ckpt      = True,
    out_dir        = OUT_DIR,
    ckpt_dir       = CKPT_DIR,
)

# DiT variant specs ────────────────────────────────────────────────────────────
DIT_SPECS = {
    'S':  dict(d_model=384,  depth=12, n_heads=6),
    'B':  dict(d_model=768,  depth=12, n_heads=12),
    'L':  dict(d_model=1024, depth=24, n_heads=16),
    'XL': dict(d_model=1152, depth=28, n_heads=16),
}
CFG['dit_spec'] = DIT_SPECS[CFG['dit_variant']]

# Reproducibility
random.seed(CFG['seed']); np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])
if DEVICE == 'cuda': torch.cuda.manual_seed_all(CFG['seed'])

d_lats = [c * CFG['latent_spatial']**2 for c in CFG['latent_channels']]
d_ints = [sn - 1 for sn in CFG['sphere_n_grid']]
print(f'DiT-{CFG["dit_variant"]} | {CFG["objective"]} | {CFG["fm_sample_steps"]} Euler steps')
print(f'Sphere dim sweep: {CFG["sphere_n_grid"]}  →  d_intrinsic = {d_ints}')
print(f'Ambient embedding: R^{CFG["d_ambient"]}')
print(f'Latent channel sweep: {CFG["latent_channels"]}')
print(f'd_latent per channel: {d_lats}')
print(f'n_train = {CFG["n_train"]:,}  (fixed)')

---
## 2 · Hypersphere data generation

In [ ]:
from torch.utils.data import DataLoader, TensorDataset


def sample_hypersphere(n_samples, n_dim, seed=42):
    """Uniformly sample n_samples points on S^{n_dim-1} ⊂ R^{n_dim}.
    Intrinsic dimension = n_dim - 1.
    """
    rng = np.random.RandomState(seed)
    x = rng.randn(n_samples, n_dim)
    x /= np.linalg.norm(x, axis=1, keepdims=True)
    return x.astype(np.float32)


def embed_ambient(data, d_ambient, seed=42):
    """Isometrically embed R^n data into R^{d_ambient} via random orthonormal map."""
    n_dim = data.shape[1]
    if d_ambient == n_dim:
        return data
    assert d_ambient > n_dim, 'd_ambient must be ≥ data dimension'
    rng = np.random.RandomState(seed)
    Q, _ = np.linalg.qr(rng.randn(d_ambient, d_ambient))
    P = Q[:, :n_dim]                           # (d_ambient, n_dim) orthonormal cols
    return (data @ P.T).astype(np.float32)      # (n, d_ambient)


def generate_sphere_data(sphere_n):
    """Generate hypersphere train+test loaders for a given sphere dimension.

    Updates global train_loader / test_loader so downstream VAE training
    functions pick up the new data automatically.
    """
    global train_loader, test_loader

    d_int = sphere_n - 1
    n_train = CFG['n_train']

    print(f'Sampling S^{d_int} ⊂ R^{sphere_n}  (d_intrinsic={d_int})')
    train_sphere = sample_hypersphere(n_train, sphere_n, seed=CFG['seed'])
    test_sphere  = sample_hypersphere(CFG['n_test'], sphere_n,
                                       seed=CFG['seed'] + 99999)

    if CFG['d_ambient'] > sphere_n:
        print(f'  Embedding into R^{CFG["d_ambient"]} ...', flush=True)
        train_data = embed_ambient(train_sphere, CFG['d_ambient'], seed=CFG['seed'])
        test_data  = embed_ambient(test_sphere,  CFG['d_ambient'], seed=CFG['seed'])
    else:
        train_data = train_sphere
        test_data  = test_sphere

    train_ds = TensorDataset(
        torch.from_numpy(train_data),
        torch.zeros(len(train_data), dtype=torch.long),
    )
    test_ds = TensorDataset(
        torch.from_numpy(test_data),
        torch.zeros(len(test_data), dtype=torch.long),
    )
    train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                              num_workers=0, pin_memory=(DEVICE == 'cuda'),
                              drop_last=True)
    test_loader  = DataLoader(test_ds, batch_size=CFG['batch_size'], shuffle=False,
                              num_workers=0, pin_memory=(DEVICE == 'cuda'),
                              drop_last=False)

    print(f'  Train: {len(train_ds):,}  |  Test: {len(test_ds):,}  '
          f'|  d_ambient={CFG["d_ambient"]}')
    return train_loader, test_loader


# Quick test with first sphere_n
train_loader, test_loader = generate_sphere_data(CFG['sphere_n_grid'][0])
print(f'Batches per epoch: {len(train_loader):,}')

---
## 3 · Diffusion schedule

In [ ]:
import math


class FlowSchedule:
    """
    Conditional flow matching with linear interpolation path.
    x_t = (1-t)*x_0 + t*noise,  velocity = noise - x_0
    t=0 → clean data,  t=1 → pure noise.
    """
    def interpolate(self, x0: torch.Tensor, t: torch.Tensor, noise=None):
        """Interpolate between data and noise; return (x_t, velocity)."""
        if noise is None:
            noise = torch.randn_like(x0)
        t_ = t.view(-1, *([1] * (x0.dim() - 1)))
        x_t = (1 - t_) * x0 + t_ * noise
        velocity = noise - x0
        return x_t, velocity


schedule = FlowSchedule()
print(f'Flow matching schedule ready  (linear interpolation, t ∈ [0, 1])')

---
## 4 · DiT model

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange

# ── Timestep sinusoidal embedding ─────────────────────────────────────────────
class SinusoidalEmbed(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        assert dim % 2 == 0
        self.dim = dim

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        half = self.dim // 2
        freq = torch.exp(-math.log(10000) *
                         torch.arange(half, device=t.device) / (half - 1))
        emb  = (t.float() * 1000).unsqueeze(1) * freq.unsqueeze(0)
        return torch.cat([emb.sin(), emb.cos()], dim=-1)   # (B, dim)


# ── AdaLN-Zero modulation ─────────────────────────────────────────────────────
class AdaLNZero(nn.Module):
    """
    Conditioning via Adaptive LayerNorm with zero-init final projection.
    Produces scale (γ) and shift (β) from conditioning vector c.
    """
    def __init__(self, d_model: int, d_cond: int, n_params: int = 6):
        super().__init__()
        self.norm   = nn.LayerNorm(d_model, elementwise_affine=False, eps=1e-6)
        self.proj   = nn.Linear(d_cond, n_params * d_model)
        # Zero-init so network starts as identity
        nn.init.zeros_(self.proj.weight)
        nn.init.zeros_(self.proj.bias)
        self.n_params = n_params

    def forward(self, x: torch.Tensor, c: torch.Tensor):
        """Returns normalised x and a list of n_params modulation tensors."""
        params = self.proj(F.silu(c))   # (B, n_params * d_model)
        params = params.chunk(self.n_params, dim=-1)   # n_params × (B, d_model)
        return self.norm(x), params


# ── Multi-Head Self-Attention ──────────────────────────────────────────────────
class MHSA(nn.Module):
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv  = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model)
        self.scale = self.head_dim ** -0.5

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(2)          # each: (B, N, H, D)
        q = q.transpose(1, 2)            # (B, H, N, D)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        # Use scaled_dot_product_attention if available (torch ≥ 2.0)
        if hasattr(F, 'scaled_dot_product_attention'):
            out = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0)
        else:
            attn = (q @ k.transpose(-2, -1)) * self.scale
            attn = attn.softmax(dim=-1)
            out  = attn @ v
        out = out.transpose(1, 2).reshape(B, N, C)
        return self.proj(out)


# ── DiT Block ─────────────────────────────────────────────────────────────────
class DiTBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_cond: int, mlp_ratio: float = 4.0):
        super().__init__()
        self.adaLN = AdaLNZero(d_model, d_cond, n_params=6)
        self.attn  = MHSA(d_model, n_heads)
        mlp_hidden = int(d_model * mlp_ratio)
        self.mlp   = nn.Sequential(
            nn.Linear(d_model, mlp_hidden),
            nn.GELU(),
            nn.Linear(mlp_hidden, d_model),
        )

    def forward(self, x: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
        x_norm, (γ1, β1, α1, γ2, β2, α2) = self.adaLN(x, c)
        x = x + α1.unsqueeze(1) * self.attn(
            (1 + γ1.unsqueeze(1)) * x_norm + β1.unsqueeze(1))
        x_norm2 = nn.functional.layer_norm(x, x.shape[-1:])
        x = x + α2.unsqueeze(1) * self.mlp(
            (1 + γ2.unsqueeze(1)) * x_norm2 + β2.unsqueeze(1))
        return x


# ── Full DiT ──────────────────────────────────────────────────────────────────
class DiT(nn.Module):
    """
    Diffusion Transformer following Peebles & Xie 2023.

    Args:
        in_channels : latent channels c
        spatial_size: latent h = w (e.g. 4 → d_latent = c × 16)
        patch_size  : p (patchify the latent spatial grid)
        d_model, depth, n_heads: DiT variant dimensions
        n_classes   : number of conditioning classes (0 = unconditional)
    """
    def __init__(
        self,
        in_channels : int,
        spatial_size: int,
        patch_size  : int,
        d_model     : int,
        depth       : int,
        n_heads     : int,
        n_classes   : int = 0,
        d_cond      : int = None,
    ):
        super().__init__()
        self.in_channels  = in_channels
        self.spatial_size = spatial_size
        self.patch_size   = patch_size
        self.n_patches    = (spatial_size // patch_size) ** 2
        self.patch_dim    = in_channels * patch_size * patch_size
        d_cond            = d_cond or d_model

        # ── Input ────────────────────────────────────────────────────────────
        self.patch_embed = nn.Linear(self.patch_dim, d_model)
        self.pos_embed   = nn.Parameter(
            torch.zeros(1, self.n_patches, d_model))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        # ── Conditioning ─────────────────────────────────────────────────────
        self.t_embed = nn.Sequential(
            SinusoidalEmbed(d_model),
            nn.Linear(d_model, d_cond),
            nn.SiLU(),
            nn.Linear(d_cond, d_cond),
        )
        if n_classes > 0:
            self.class_embed = nn.Embedding(n_classes + 1, d_cond)  # +1 for null
        else:
            self.class_embed = None

        # ── Transformer blocks ────────────────────────────────────────────────
        self.blocks = nn.ModuleList([
            DiTBlock(d_model, n_heads, d_cond) for _ in range(depth)
        ])

        # ── Output ────────────────────────────────────────────────────────────
        self.final_norm = nn.LayerNorm(d_model, elementwise_affine=False, eps=1e-6)
        self.final_adaLN_proj = nn.Linear(d_cond, 2 * d_model)
        nn.init.zeros_(self.final_adaLN_proj.weight)
        nn.init.zeros_(self.final_adaLN_proj.bias)
        self.final_proj = nn.Linear(d_model, self.patch_dim)
        nn.init.zeros_(self.final_proj.weight)
        nn.init.zeros_(self.final_proj.bias)

    def patchify(self, z: torch.Tensor) -> torch.Tensor:
        """(B,C,H,W) → (B, n_patches, patch_dim)"""
        p = self.patch_size
        return rearrange(z, 'b c (h p1) (w p2) -> b (h w) (c p1 p2)', p1=p, p2=p)

    def unpatchify(self, x: torch.Tensor) -> torch.Tensor:
        """(B, n_patches, patch_dim) → (B,C,H,W)"""
        p  = self.patch_size
        hw = int(self.n_patches ** 0.5)
        return rearrange(x, 'b (h w) (c p1 p2) -> b c (h p1) (w p2)',
                         h=hw, w=hw, p1=p, p2=p)

    def forward(
        self,
        z_t    : torch.Tensor,          # (B, C, H, W) noisy latent
        t      : torch.Tensor,          # (B,) integer timesteps
        labels : torch.Tensor = None,   # (B,) class labels or None
    ) -> torch.Tensor:
        # Patchify + embed
        x = self.patchify(z_t)          # (B, N, patch_dim)
        x = self.patch_embed(x)         # (B, N, d_model)
        x = x + self.pos_embed

        # Conditioning vector
        c = self.t_embed(t)             # (B, d_cond)
        if self.class_embed is not None and labels is not None:
            c = c + self.class_embed(labels)

        # Transformer blocks
        for block in self.blocks:
            x = block(x, c)

        # Final AdaLN + project back to patch_dim
        shift, scale = self.final_adaLN_proj(F.silu(c)).chunk(2, dim=-1)
        x = self.final_norm(x) * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)
        x = self.final_proj(x)          # (B, N, patch_dim)
        return self.unpatchify(x)       # (B, C, H, W)


def build_dit(in_channels: int) -> DiT:
    spec = CFG['dit_spec']
    model = DiT(
        in_channels  = in_channels,
        spatial_size = CFG['latent_spatial'],
        patch_size   = CFG['patch_size'],
        d_model      = spec['d_model'],
        depth        = spec['depth'],
        n_heads      = spec['n_heads'],
        n_classes    = CFG['n_classes'],
    ).to(DEVICE)
    return model


# Quick sanity check
_c  = CFG['latent_channels'][0]
_sp = CFG['latent_spatial']
_m  = build_dit(_c)
_z  = torch.randn(2, _c, _sp, _sp, device=DEVICE)
_t  = torch.rand(2, device=DEVICE)
_out = _m(_z, _t)
print(f'DiT-{CFG["dit_variant"]} smoke test OK  '
      f'in={tuple(_z.shape)} → out={tuple(_out.shape)}')
n_params = sum(p.numel() for p in _m.parameters()) / 1e6
print(f'Parameters: {n_params:.1f} M')
del _m, _z, _t, _out

---
## 5 · EMA, DDIM sampler & FID utilities

In [ ]:
import copy
from scipy.linalg import sqrtm

# ── Exponential Moving Average ─────────────────────────────────────────────────
class EMA:
    """Maintains a shadow copy of model weights via EMA."""
    def __init__(self, model: nn.Module, decay: float = 0.9999):
        self.decay  = decay
        self.shadow = copy.deepcopy(model).eval()
        for p in self.shadow.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model: nn.Module):
        for s, m in zip(self.shadow.parameters(), model.parameters()):
            s.data.mul_(self.decay).add_(m.data, alpha=1 - self.decay)

    def __call__(self, *args, **kwargs):
        return self.shadow(*args, **kwargs)


# ── Euler ODE sampler for flow matching ───────────────────────────────────────
@torch.no_grad()
def euler_sample(
    model    : nn.Module,
    shape    : tuple,
    n_steps  : int = 50,
    labels   : torch.Tensor = None,
    cfg_scale: float = 1.0,
    x_init   : torch.Tensor = None,
) -> torch.Tensor:
    """Euler ODE sampling for flow matching (t=1 noise -> t=0 data)."""
    model.eval()
    device = next(model.parameters()).device
    dt = 1.0 / n_steps
    x = x_init.clone() if x_init is not None else torch.randn(shape, device=device)

    for i in range(n_steps):
        t_val = 1.0 - i * dt
        t_batch = torch.full((shape[0],), t_val, device=device)

        v = model(x, t_batch, labels)

        if cfg_scale > 1.0 and labels is not None:
            null_labels = torch.full_like(labels, CFG['n_classes'])
            v_null = model(x, t_batch, null_labels)
            v = v_null + cfg_scale * (v - v_null)

        x = x - v * dt

    return x


# ── FID (feature-space Fréchet distance) ─────────────────────────────────────
def compute_fid(real: np.ndarray, fake: np.ndarray, eps: float = 1e-6) -> float:
    mu_r, mu_f = real.mean(0), fake.mean(0)
    C_r = np.cov(real, rowvar=False)
    C_f = np.cov(fake, rowvar=False)
    # Regularise to avoid singular/near-singular matrices (happens at early
    # training when generated samples are nearly identical)
    C_r += np.eye(C_r.shape[0]) * eps
    C_f += np.eye(C_f.shape[0]) * eps
    diff = mu_r - mu_f
    covmean = sqrtm(C_r @ C_f).real
    # Clip any residual imaginary noise
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    return float(diff @ diff + np.trace(C_r + C_f - 2 * covmean))


def extract_features(data, n_max=5000) -> np.ndarray:
    """Flatten spatial dims into feature vectors for LFD computation."""
    if isinstance(data, np.ndarray):
        return data.reshape(len(data), -1)[:n_max]
    if isinstance(data, torch.Tensor):
        return data.cpu().numpy().reshape(len(data), -1)[:n_max]
    feats = []
    for batch in data:
        imgs = batch[0] if isinstance(batch, (list, tuple)) else batch
        feats.append(imgs.reshape(imgs.size(0), -1).cpu().numpy())
        if sum(len(f) for f in feats) >= n_max:
            break
    return np.concatenate(feats, axis=0)[:n_max]


# ── FLIPD LID via Hutchinson trace estimator (Kamkari et al. 2024) ────────────
def flipd_lid_dit(
    model   : nn.Module,
    z_0     : torch.Tensor,
    t_val   : float = 0.1,
    n_probes: int = 5,
    labels  : torch.Tensor = None,
) -> torch.Tensor:
    """LID(x,t) ≈ d − t · tr(∂ε̂/∂x_t) estimated via Hutchinson (flow matching)."""
    device = next(model.parameters()).device
    d = z_0[0].numel()

    noise = torch.randn_like(z_0)
    z_t = (1 - t_val) * z_0 + t_val * noise
    t_vec = torch.full((z_0.shape[0],), t_val, device=device)

    traces = []
    for _ in range(n_probes):
        z_in = z_t.detach().requires_grad_(True)
        v_pred = model(z_in, t_vec, labels)
        eps_pred = (1 - t_val) * v_pred + z_in
        probe = torch.randint(0, 2, z_in.shape, device=device).float() * 2 - 1
        (grad_eps,) = torch.autograd.grad(
            outputs=eps_pred, inputs=z_in,
            grad_outputs=probe, create_graph=False,
        )
        traces.append((probe * grad_eps).flatten(1).sum(dim=1))

    avg_trace = torch.stack(traces).mean(dim=0)
    lid = d - t_val * avg_trace
    return lid.detach()


# ── Nearest-neighbour memorisation ratio (Carlini / Gu criterion) ─────────────
def nn_memorization_ratio(
    generated: np.ndarray,
    training : np.ndarray,
    threshold: float = 0.5,
) -> tuple:
    """Per-sample d1/d2 ratio and overall memorisation rate."""
    from sklearn.neighbors import NearestNeighbors
    nn_model = NearestNeighbors(n_neighbors=2, metric='euclidean').fit(training)
    dists, _ = nn_model.kneighbors(generated)
    d1, d2 = dists[:, 0], dists[:, 1]
    ratios = d1 / np.maximum(d2, 1e-10)
    return ratios, float((ratios < threshold).mean())


# ── Eigenvalue / spectral-gap analysis of latent covariance ───────────────────
def eigenvalue_analysis(data: np.ndarray) -> dict:
    """Eigenvalues of sample covariance + spectral gap statistics."""
    cov = np.cov(data, rowvar=False)
    eigs = np.sort(np.linalg.eigvalsh(cov))[::-1].astype(np.float64)
    if len(eigs) > 1:
        gaps = eigs[:-1] - eigs[1:]
        gap_idx = int(np.argmax(gaps))
        gap_val = float(gaps[gap_idx])
    else:
        gap_idx, gap_val = 0, 0.0
    effective_rank = float(np.sum(eigs > eigs[0] * 0.01))
    return {'eigenvalues': eigs, 'spectral_gap': gap_val,
            'gap_position': gap_idx, 'effective_rank': effective_rank}


print('EMA, Euler sampler, LFD, FLIPD, NN-memo, eigenvalue utilities defined.')

---
## 6 · Conv-VAE encoder/decoder

> A convolutional VAE reshapes the ambient-space sphere data into pseudo-images
> and compresses to latent tensors of shape `(c, spatial, spatial)` for DiT.
> One VAE is trained per latent channel count.

In [ ]:
from tqdm.auto import tqdm


class ConvVAE(nn.Module):
    """
    Convolutional VAE for flat vector data.
    Reshapes R^{d_input} → (in_c, 2S, 2S) pseudo-image, encodes to (c, S, S)
    latent via stride-2 convolutions, and decodes back to R^{d_input}.
    Requires d_input divisible by (2 * latent_spatial)².
    """

    def __init__(self, d_input: int, latent_channels: int, latent_spatial: int):
        super().__init__()
        self.c  = latent_channels
        self.sp = latent_spatial
        img_spatial = latent_spatial * 2
        in_c = d_input // (img_spatial ** 2)
        assert in_c * img_spatial ** 2 == d_input, \
            f'd_input={d_input} must be divisible by (2*latent_spatial)²={img_spatial**2}'
        self.in_c = in_c
        self.img_spatial = img_spatial

        c = latent_channels
        self.encoder = nn.Sequential(
            nn.Conv2d(in_c, 64,  3, 1, 1), nn.GroupNorm(8, 64),  nn.SiLU(),
            nn.Conv2d(64,   128, 4, 2, 1), nn.GroupNorm(8, 128), nn.SiLU(),
        )
        self.mu_conv = nn.Conv2d(128, c, 1)
        self.lv_conv = nn.Conv2d(128, c, 1)

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(c,  128, 4, 2, 1), nn.GroupNorm(8, 128), nn.SiLU(),
            nn.Conv2d(128, 64, 3, 1, 1),           nn.GroupNorm(8, 64),  nn.SiLU(),
            nn.Conv2d(64,  in_c, 3, 1, 1),
        )

    def _to_img(self, x):
        return x.view(-1, self.in_c, self.img_spatial, self.img_spatial)

    def _to_flat(self, img):
        return img.flatten(1)

    def encode(self, x):
        h  = self.encoder(self._to_img(x))
        mu = self.mu_conv(h)
        lv = self.lv_conv(h).clamp(-6, 6)
        return mu, lv

    def reparameterize(self, mu, lv):
        return mu + torch.randn_like(mu) * (0.5 * lv).exp()

    def decode(self, z):
        return self._to_flat(self.decoder(z))

    def forward(self, x):
        mu, lv = self.encode(x)
        z      = self.reparameterize(mu, lv)
        recon  = self.decode(z)
        recon_loss = F.mse_loss(recon, x, reduction='sum') / x.shape[0]
        kl     = -0.5 * (1 + lv - mu.pow(2) - lv.exp()).sum(dim=[1, 2, 3]).mean()
        return recon, recon_loss + CFG['vae_kl_weight'] * kl


def train_vae(latent_channels: int, n_epochs: int = None) -> ConvVAE:
    """Train a ConvVAE for a given latent channel count."""
    if n_epochs is None:
        n_epochs = CFG['vae_epochs']
    vae = ConvVAE(CFG['d_ambient'], latent_channels,
                  CFG['latent_spatial']).to(DEVICE)
    opt = torch.optim.AdamW(vae.parameters(), lr=3e-4)
    total_batches = n_epochs * len(train_loader)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total_batches)
    vae.train()
    pbar = tqdm(total=total_batches, desc=f'  VAE c={latent_channels}', leave=False)
    for epoch in range(n_epochs):
        epoch_loss = 0.0
        for batch in train_loader:
            x = batch[0].to(DEVICE)
            _, loss = vae(x)
            opt.zero_grad(); loss.backward(); opt.step()
            sched.step()
            epoch_loss += loss.item()
            pbar.update(1)
            pbar.set_postfix(loss=f'{loss.item():.4f}')
        avg = epoch_loss / len(train_loader)
        print(f'  VAE epoch {epoch+1}/{n_epochs}  avg_loss={avg:.4f}', flush=True)
    pbar.close()
    return vae


def prepare_vae_and_latents(latent_c: int, sphere_n: int):
    """Train or load VAE, encode train+test data, return everything."""
    d_latent = latent_c * CFG['latent_spatial'] ** 2
    vae_file     = f'vae_c{latent_c}_sn{sphere_n}.pt'
    latents_file = f'latents_c{latent_c}_sn{sphere_n}.pt'

    vae_path = latents_path = None
    for search_dir in filter(None, [INPUT_CKPT_DIR, CFG['ckpt_dir']]):
        v = os.path.join(search_dir, vae_file)
        l = os.path.join(search_dir, latents_file)
        if os.path.exists(v) and os.path.exists(l):
            vae_path, latents_path = v, l
            break

    if vae_path is not None and latents_path is not None:
        print(f'\n{"="*60}', flush=True)
        print(f'  Loading cached VAE + latents: c={latent_c}  '
              f'(d_latent={d_latent})', flush=True)
        print(f'{"="*60}', flush=True)
        vae = ConvVAE(CFG['d_ambient'], latent_c,
                      CFG['latent_spatial']).to(DEVICE)
        vae.load_state_dict(torch.load(vae_path, weights_only=True,
                                       map_location=DEVICE))
        vae.eval()
        cached = torch.load(latents_path, weights_only=False, map_location='cpu')
        Z_train  = cached['Z_train']
        Y_train  = cached['Y_train']
        Z_test   = cached['Z_test']
        Y_test   = cached['Y_test']
        eig_info = cached['eig_info']
        print(f'  Spectral gap: {eig_info["spectral_gap"]:.4f}  '
              f'Effective rank: {eig_info["effective_rank"]:.0f}', flush=True)
        return vae, Z_train, Y_train, Z_test, Y_test, eig_info

    print(f'\n{"="*60}', flush=True)
    print(f'  Training VAE: c={latent_c}  →  d_latent={d_latent}', flush=True)
    print(f'{"="*60}', flush=True)

    vae = train_vae(latent_c)
    vae.eval()

    @torch.no_grad()
    def encode_dataset(loader):
        latents, labels_list = [], []
        for x, y in loader:
            mu, _ = vae.encode(x.to(DEVICE))
            latents.append(mu.cpu())
            labels_list.append(y)
        return torch.cat(latents), torch.cat(labels_list)

    print('Encoding train set...', flush=True)
    Z_train, Y_train = encode_dataset(train_loader)
    print('Encoding test set...', flush=True)
    Z_test, Y_test = encode_dataset(test_loader)

    z_std = Z_train.std() + 1e-8
    Z_train, Z_test = Z_train / z_std, Z_test / z_std

    eig_info = eigenvalue_analysis(Z_train.cpu().numpy().reshape(len(Z_train), -1))
    print(f'  Spectral gap: {eig_info["spectral_gap"]:.4f}  '
          f'Effective rank: {eig_info["effective_rank"]:.0f}', flush=True)

    torch.save(vae.state_dict(), os.path.join(CFG['ckpt_dir'], vae_file))
    torch.save({'Z_train': Z_train, 'Y_train': Y_train,
                'Z_test': Z_test, 'Y_test': Y_test,
                'eig_info': eig_info},
               os.path.join(CFG['ckpt_dir'], latents_file))
    print(f'  Saved VAE + latents to {CFG["ckpt_dir"]}', flush=True)

    return vae, Z_train, Y_train, Z_test, Y_test, eig_info


print('ConvVAE defined. Will train one per latent channel count.')

---
## 7 · Training loop with mid-run evaluation

In [ ]:
from tqdm.auto import tqdm
from torch.amp import GradScaler, autocast


def run_experiment(latent_c, d_int, Z_train, Y_train,
                   Z_test, Y_test, vae=None,
                   real_ambient_test=None, real_ambient_train=None,
                   real_feats_test=None):
    """Train DiT and evaluate.

    Metrics computed:
      LFD  – latent-space Fréchet distance
      AFD  – ambient-space Fréchet distance (decode through VAE)
      memo – NN memorisation rate (ambient space when VAE available)
    """
    d_latent = latent_c * CFG['latent_spatial'] ** 2
    n_train = len(Z_train)

    torch.manual_seed(CFG['seed'] + d_latent * 100 + d_int)
    np.random.seed(CFG['seed'] + d_latent * 100 + d_int)

    print(f'\n  DiT: c={latent_c}, d_int={d_int}  (d_latent={d_latent})', flush=True)

    from torch.utils.data import TensorDataset as TDS
    latent_loader = torch.utils.data.DataLoader(
        TDS(Z_train, Y_train),
        batch_size=min(CFG['batch_size'], n_train),
        shuffle=True, pin_memory=(DEVICE == 'cuda'), drop_last=True,
    )

    model = build_dit(latent_c)
    ema   = EMA(model, decay=CFG['ema_decay'])
    opt   = torch.optim.AdamW(model.parameters(), lr=CFG['lr'],
                               betas=(0.9, 0.999), weight_decay=0.0)
    scaler = GradScaler('cuda', enabled=(CFG['mixed_prec'] and DEVICE=='cuda'))

    history = dict(
        steps=[],
        loss_steps=[], loss_vals=[],
        lfd_train=[], lfd_test=[],
        afd_test=[],
        memo_rate=[],
        flipd={t: {'mean': [], 'std': []} for t in CFG['flipd_t_vals']},
        gen_eff_rank=[],
        tau_gen=None, tau_mem=None,
    )

    real_feats_train = Z_train[:CFG['n_fid_samples']].cpu().numpy().reshape(
        min(CFG['n_fid_samples'], len(Z_train)), -1)
    if real_feats_test is None:
        real_feats_test = Z_test[:CFG['n_fid_samples']].cpu().numpy().reshape(
            min(CFG['n_fid_samples'], len(Z_test)), -1)

    step = 0
    loader_iter = iter(latent_loader)
    pbar = tqdm(total=CFG['total_steps'],
                desc=f'c={latent_c} d_int={d_int}', leave=False)

    while step < CFG['total_steps']:
        model.train()
        try:
            z0_batch, lbl_batch = next(loader_iter)
        except StopIteration:
            loader_iter = iter(latent_loader)
            z0_batch, lbl_batch = next(loader_iter)

        z0    = z0_batch.to(DEVICE)
        lbls  = lbl_batch.to(DEVICE) if CFG['n_classes'] > 0 else None
        t     = torch.rand(z0.size(0), device=DEVICE)
        noise = torch.randn_like(z0)
        z_t, velocity = schedule.interpolate(z0, t, noise)

        with autocast('cuda', enabled=(CFG['mixed_prec'] and DEVICE=='cuda')):
            v_pred = model(z_t, t, lbls)
            loss   = F.mse_loss(v_pred, velocity)

        opt.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
        scaler.step(opt); scaler.update()
        ema.update(model)
        step += 1
        pbar.update(1)
        pbar.set_postfix(loss=f'{loss.item():.4f}')

        if step % CFG['loss_every'] == 0:
            history['loss_steps'].append(step)
            history['loss_vals'].append(loss.item())

        # ── Periodic evaluation ───────────────────────────────────────────────
        if step % CFG['eval_every'] == 0 or step == CFG['total_steps']:
            ema.shadow.eval()
            n_samp = CFG['n_fid_samples']

            # Generate samples in batches
            fake_batches = []
            remaining = n_samp
            while remaining > 0:
                bs = min(256, remaining)
                s  = (bs, latent_c, CFG['latent_spatial'], CFG['latent_spatial'])
                lbl_s = (torch.randint(0, CFG['n_classes'], (bs,), device=DEVICE)
                         if CFG['n_classes'] > 0 else None)
                fake_batches.append(
                    euler_sample(ema.shadow, s,
                                 n_steps=CFG['fm_sample_steps'], labels=lbl_s).cpu()
                )
                remaining -= bs
            fake_z = torch.cat(fake_batches)[:n_samp]
            fake_feats = fake_z.cpu().numpy().reshape(len(fake_z), -1)

            lfd_tr = lfd_te = float('nan')
            try:
                lfd_tr = compute_fid(real_feats_train[:len(fake_feats)], fake_feats)
                lfd_te = compute_fid(real_feats_test[:len(fake_feats)], fake_feats)
            except Exception as e:
                print(f'  LFD error at step {step}: {e}', flush=True)

            # Ambient-space Fréchet distance (decode latents → R^d_ambient)
            afd_te = float('nan')
            fake_ambient = None
            if vae is not None and real_ambient_test is not None:
                try:
                    vae.eval()
                    with torch.no_grad():
                        decoded = []
                        for di in range(0, len(fake_z), 256):
                            decoded.append(
                                vae.decode(fake_z[di:di+256].to(DEVICE)).cpu())
                        fake_ambient = torch.cat(decoded).cpu().numpy()
                    afd_te = compute_fid(real_ambient_test, fake_ambient)
                except Exception as e:
                    print(f'  AFD error at step {step}: {e}', flush=True)

            # NN memorisation in ambient space when available
            if fake_ambient is not None and real_ambient_train is not None:
                _, memo_rate = nn_memorization_ratio(
                    fake_ambient, real_ambient_train,
                    threshold=CFG['memo_nn_thresh'])
            else:
                _, memo_rate = nn_memorization_ratio(
                    fake_feats, real_feats_train,
                    threshold=CFG['memo_nn_thresh'])

            # FLIPD via Hutchinson trace on DiT (multiple t values)
            n_flipd = min(32, len(Z_train))
            flipd_idx = torch.randperm(len(Z_train))[:n_flipd]
            flipd_z = Z_train[flipd_idx].to(DEVICE)
            flipd_y = (Y_train[flipd_idx].to(DEVICE)
                       if CFG['n_classes'] > 0 else None)
            for t_val in CFG['flipd_t_vals']:
                try:
                    lid_vals = flipd_lid_dit(
                        ema.shadow, flipd_z,
                        t_val=t_val,
                        n_probes=CFG['flipd_n_probes'],
                        labels=flipd_y)
                    history['flipd'][t_val]['mean'].append(float(lid_vals.mean().item()))
                    history['flipd'][t_val]['std'].append(float(lid_vals.std().item()))
                except Exception as e:
                    print(f'  FLIPD error at step {step}, t={t_val}: {e}')
                    history['flipd'][t_val]['mean'].append(float('nan'))
                    history['flipd'][t_val]['std'].append(float('nan'))

            # Eigenvalue analysis on generated samples
            gen_rank = float('nan')
            try:
                gen_eig = eigenvalue_analysis(fake_feats)
                gen_rank = gen_eig['effective_rank']
            except Exception as e:
                print(f'  Gen eigenvalue error at step {step}: {e}', flush=True)

            history['steps'].append(step)
            history['lfd_train'].append(lfd_tr)
            history['lfd_test'].append(lfd_te)
            history['afd_test'].append(afd_te)
            history['memo_rate'].append(memo_rate)
            history['gen_eff_rank'].append(gen_rank)

            # tau_gen: use AFD when available, fall back to LFD
            fid_for_tau = afd_te if not np.isnan(afd_te) else lfd_te
            if history['tau_gen'] is None and fid_for_tau < CFG['fid_threshold']:
                history['tau_gen'] = step
                metric_used = 'AFD' if not np.isnan(afd_te) else 'LFD'
                print(f"  [Step {step}] Generation onset (tau_gen) via {metric_used}! "
                      f"({fid_for_tau:.2f} < {CFG['fid_threshold']})", flush=True)

            if history['tau_mem'] is None and memo_rate > CFG['memo_threshold']:
                history['tau_mem'] = step
                print(f"  [Step {step}] Memorization onset (tau_mem)! "
                      f"(Rate: {memo_rate:.3f} > {CFG['memo_threshold']})", flush=True)

            _flipd_str = '  '.join(
                f't{t}={history["flipd"][t]["mean"][-1]:.1f}'
                for t in CFG['flipd_t_vals'])
            print(f'  step={step:6d}  AFD={afd_te:.1f}  '
                  f'LFD_te={lfd_te:.1f}  memo={memo_rate:.3f}  '
                  f'FLIPD[{_flipd_str}]  '
                  f'gen_rank={gen_rank:.0f}', flush=True)

            if CFG['save_ckpt'] and step == CFG['total_steps']:
                ckpt_path = os.path.join(
                    CFG['ckpt_dir'],
                    f'dit_c{latent_c}_dint{d_int}_step{step}.pt')
                torch.save({'model': ema.shadow.state_dict(),
                            'step': step, 'history': history}, ckpt_path)

    pbar.close()
    print(f'  τ_gen={history["tau_gen"]}  τ_mem={history["tau_mem"]}', flush=True)
    return history


print('Training functions defined.')

---
## 8 · Run the latent dim sweep

In [ ]:
# ── Outer sweep: sphere_n × latent_channels ──────────────────────────────────
all_histories = {}
eig_cache = {}
vae_baseline_cache = {}

for sphere_n in CFG['sphere_n_grid']:
    d_int = sphere_n - 1
    print(f'\n{"="*60}')
    print(f'  Sphere S^{d_int}: d_intrinsic = {d_int}')
    print(f'{"="*60}')

    generate_sphere_data(sphere_n)

    # Ambient-space references for AFD + memorisation (per sphere_n)
    _raw_test  = torch.cat([b[0] for b in test_loader])[:CFG['n_fid_samples']]
    _raw_train = torch.cat([b[0] for b in train_loader])
    real_ambient_test  = _raw_test.cpu().numpy()
    real_ambient_train = _raw_train.cpu().numpy()

    for c in CFG['latent_channels']:
        vae, Z_tr, Y_tr, Z_te, Y_te, eig_info = prepare_vae_and_latents(c, sphere_n)
        eig_cache[(c, d_int)] = eig_info

        real_feats_test_c = Z_te[:CFG['n_fid_samples']].cpu().numpy().reshape(
            min(CFG['n_fid_samples'], len(Z_te)), -1)

        # VAE baseline AFD: real → encode → decode → AFD vs real
        vae.eval()
        with torch.no_grad():
            recon_parts = []
            for ri in range(0, len(_raw_test), 256):
                mu, _ = vae.encode(_raw_test[ri:ri+256].to(DEVICE))
                recon_parts.append(vae.decode(mu).cpu())
            recon_ambient = torch.cat(recon_parts).cpu().numpy()
        vae_baseline_afd = compute_fid(real_ambient_test, recon_ambient)
        vae_baseline_cache[(c, d_int)] = vae_baseline_afd
        print(f'  VAE baseline AFD (c={c}, d_int={d_int}): {vae_baseline_afd:.1f}',
              flush=True)

        hist = run_experiment(c, d_int, Z_tr, Y_tr, Z_te, Y_te,
                              vae=vae,
                              real_ambient_test=real_ambient_test,
                              real_ambient_train=real_ambient_train,
                              real_feats_test=real_feats_test_c)
        all_histories[(c, d_int)] = hist

# ── Save this run's results ──────────────────────────────────────────────────
run_save_path = os.path.join(OUT_DIR, f'results_run{RUN_ID}.pt')
_save_histories = {}
for (c, d_int), h in all_histories.items():
    _save_histories[f'{c}_{d_int}'] = h
_save_eig = {}
for (c, d_int), e in eig_cache.items():
    _save_eig[f'{c}_{d_int}'] = e
_save_vae_bl = {}
for (c, d_int), v in vae_baseline_cache.items():
    _save_vae_bl[f'{c}_{d_int}'] = v
torch.save({
    'all_histories': _save_histories,
    'eig_cache': _save_eig,
    'vae_baseline_cache': _save_vae_bl,
    'run_id': RUN_ID,
    'sphere_n_grid': CFG['sphere_n_grid'],
}, run_save_path)
print(f'\nRun {RUN_ID} results saved → {run_save_path}')

_flipd_hdr = '  '.join(f'FLIPD_{t}' for t in CFG['flipd_t_vals'])
print(f'\n{"c":>4}  {"d_lat":>6}  {"d_int":>6}  {"tau_gen":>8}  '
      f'{"tau_mem":>8}  {"AFD_te":>8}  {"LFD_te":>8}  {"memo":>6}  {_flipd_hdr}')
for (c, d_int), h in all_histories.items():
    d = c * CFG['latent_spatial']**2
    best_afd = min(h['afd_test']) if h.get('afd_test') else float('nan')
    best_lfd = min(h['lfd_test']) if h['lfd_test'] else float('nan')
    final_memo = h['memo_rate'][-1] if h['memo_rate'] else float('nan')
    flipd_strs = []
    for t_val in CFG['flipd_t_vals']:
        fdata = h.get('flipd', {}).get(t_val, {}).get('mean', [])
        flipd_strs.append(f'{fdata[-1]:>8.1f}' if fdata else f'{"nan":>8}')
    print(f'{c:>4}  {d:>6}  {d_int:>6}  '
          f'{str(h["tau_gen"]):>8}  {str(h["tau_mem"]):>8}  '
          f'{best_afd:>8.1f}  {best_lfd:>8.1f}  '
          f'{final_memo:>6.3f}  {"  ".join(flipd_strs)}')

---
## 8b · Merge results from all runs
> Run this cell **after all Kaggle sessions** are complete.
> Upload `results_run1.pt`, `results_run2.pt`, `results_run3.pt` from each session's output
> into `OUT_DIR` (or `/kaggle/working/synthetic/`), then execute.
> Each run covers different sphere dimensions (d_intrinsic values).

In [ ]:
# ── Merge results from all runs ───────────────────────────────────────────────
import glob as _glob

merged_histories = {}
merged_eig_cache = {}
merged_vae_baseline = {}

run_files = sorted(_glob.glob(os.path.join(OUT_DIR, 'results_run*.pt')))
print(f'Found {len(run_files)} result file(s): {[os.path.basename(f) for f in run_files]}')

for rf in run_files:
    data = torch.load(rf, weights_only=False, map_location='cpu')
    for key_str, hist in data['all_histories'].items():
        c, d_int = int(key_str.split('_')[0]), int(key_str.split('_')[1])
        if 'flipd_mean' in hist and 'flipd' not in hist:
            default_t = CFG['flipd_primary_t']
            hist['flipd'] = {default_t: {'mean': hist.pop('flipd_mean'),
                                          'std': hist.pop('flipd_std')}}
        merged_histories[(c, d_int)] = hist
    for key_str, eig in data['eig_cache'].items():
        c, d_int = int(key_str.split('_')[0]), int(key_str.split('_')[1])
        merged_eig_cache[(c, d_int)] = eig
    for key_str, afd_val in data.get('vae_baseline_cache', {}).items():
        c, d_int = int(key_str.split('_')[0]), int(key_str.split('_')[1])
        merged_vae_baseline[(c, d_int)] = afd_val
    print(f'  Loaded run {data["run_id"]}: sphere_n={data.get("sphere_n_grid", "?")}')

all_histories = merged_histories
eig_cache = merged_eig_cache
vae_baseline_cache = merged_vae_baseline

print(f'\nMerged grid: {len(all_histories)} experiments')
for (c, d_int) in sorted(all_histories.keys()):
    d = c * CFG['latent_spatial']**2
    h = all_histories[(c, d_int)]
    best_afd = min(h.get('afd_test', [float('nan')])) if h.get('afd_test') else float('nan')
    print(f'  c={c:>2}  d_lat={d:>4}  d_int={d_int:>3}  best_AFD={best_afd:.1f}  '
          f'tau_gen={h.get("tau_gen")}  tau_mem={h.get("tau_mem")}')

---
## 9 · Results — full visualisation dashboard

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False,
                     'font.size': 10})

cs       = CFG['latent_channels']
d_ints   = sorted(set(d for (_, d) in all_histories.keys()))
d_int_max = max(d_ints) if d_ints else 0
d_lats   = [c * CFG['latent_spatial']**2 for c in cs]
cmap_c   = plt.cm.plasma(np.linspace(0.1, 0.9, len(cs)))


def H(c, d_int=None):
    """Access history for (c, d_int); defaults to largest d_intrinsic."""
    if d_int is None:
        d_int = d_int_max
    h = all_histories.get((c, d_int))
    if h:
        return h
    return dict(steps=[],
                loss_steps=[], loss_vals=[],
                lfd_train=[], lfd_test=[], afd_test=[],
                memo_rate=[],
                flipd={t: {'mean': [], 'std': []} for t in CFG['flipd_t_vals']},
                gen_eff_rank=[],
                tau_gen=None, tau_mem=None)


fig = plt.figure(figsize=(20, 23))
gs  = gridspec.GridSpec(5, 3, hspace=0.50, wspace=0.38,
                        left=0.06, right=0.97, top=0.96, bottom=0.03)

# ── (a) AFD (ambient Fréchet distance) vs steps ─────────────────────────────
ax = fig.add_subplot(gs[0, 0])
for i, c in enumerate(cs):
    h = H(c)
    if not h.get('afd_test'): continue
    ax.plot(h['steps'], h['afd_test'], color=cmap_c[i], lw=2,
            label=f'c={c} (d={d_lats[i]})')
    if h['tau_gen']:
        ax.axvline(h['tau_gen'], color=cmap_c[i], ls=':', lw=1, alpha=0.7)
        ax.annotate('τ_gen', xy=(h['tau_gen'], 0.97),
                    xycoords=('data', 'axes fraction'), fontsize=6,
                    color=cmap_c[i], rotation=90, ha='right', va='top')
    bl = vae_baseline_cache.get((c, d_int_max))
    if bl is not None:
        ax.axhline(bl, color=cmap_c[i], ls='--', lw=1, alpha=0.4)
ax.axhline(CFG['fid_threshold'], color='#6b7280', ls='--', lw=1.5,
           label=f'τ_gen thresh ({CFG["fid_threshold"]})')
ax.set_title(f'(a) AFD vs steps  (d_int={d_int_max})\n'
             f'(dashed = VAE reconstruction baseline)', fontweight='bold',
             fontsize=9)
ax.set_xlabel('Training steps'); ax.set_ylabel('AFD (ambient Fréchet distance)')
ax.legend(fontsize=7, ncol=1); ax.grid(alpha=0.2)

# ── (b) LFD (latent Fréchet distance) vs steps ──────────────────────────────
ax = fig.add_subplot(gs[0, 1])
for i, c in enumerate(cs):
    h = H(c)
    if not h['lfd_test']: continue
    ax.plot(h['steps'], h['lfd_test'], color=cmap_c[i], lw=2,
            label=f'c={c} (d={d_lats[i]})')
ax.set_title(f'(b) LFD vs steps  (d_int={d_int_max})', fontweight='bold')
ax.set_xlabel('Training steps'); ax.set_ylabel('LFD (latent space)')
ax.legend(fontsize=7, ncol=1); ax.grid(alpha=0.2)

# ── (c) NN memorisation rate over training steps ─────────────────────────────
ax = fig.add_subplot(gs[0, 2])
for i, c in enumerate(cs):
    h = H(c)
    if not h['memo_rate']: continue
    ax.plot(h['steps'], h['memo_rate'], color=cmap_c[i], lw=2, label=f'c={c}')
    if h['tau_mem']:
        ax.axvline(h['tau_mem'], color=cmap_c[i], ls=':', lw=1, alpha=0.7)
ax.axhline(CFG['memo_threshold'], color='#dc2626', ls='--', lw=1.5,
           label=f'τ_mem thresh ({CFG["memo_threshold"]})')
ax.set_title(f'(c) NN memo rate vs steps  (d_int={d_int_max})', fontweight='bold')
ax.set_xlabel('Training steps'); ax.set_ylabel('Memo rate (d₁/d₂ < thresh)')
ax.legend(fontsize=7); ax.grid(alpha=0.2)

# ── (d) Eigenvalue spectra of latent space ────────────────────────────────────
ax = fig.add_subplot(gs[1, 0])
for i, c in enumerate(cs):
    eig = eig_cache.get((c, d_int_max))
    if eig is None: continue
    eigvals = eig['eigenvalues']
    n_show = min(50, len(eigvals))
    ax.semilogy(range(1, n_show + 1), eigvals[:n_show],
                color=cmap_c[i], lw=1.8, marker='.', ms=4,
                label=f'c={c} (rank={eig["effective_rank"]:.0f})')
    gap_pos = eig['gap_position']
    if 0 < gap_pos < n_show:
        ax.axvline(gap_pos + 1, color=cmap_c[i], ls=':', lw=1, alpha=0.6)
ax.set_title('(d) Eigenvalue spectra of latent covariance', fontweight='bold')
ax.set_xlabel('Eigenvalue index'); ax.set_ylabel('Eigenvalue (log scale)')
ax.legend(fontsize=7); ax.grid(alpha=0.2)

# ── (e) Best AFD vs d_intrinsic (one line per d_latent) ───────────────────────
ax = fig.add_subplot(gs[1, 1])
for i, c in enumerate(cs):
    afds_by_dint = []
    for di in d_ints:
        h = H(c, di)
        afd_vals = h.get('afd_test', [])
        afd_vals = [v for v in afd_vals if not np.isnan(v)]
        afds_by_dint.append(min(afd_vals) if afd_vals else float('nan'))
    ax.plot(d_ints, afds_by_dint, 'o-', color=cmap_c[i], lw=2, ms=6,
            label=f'd_lat={d_lats[i]}')
ax.set_title('(e) Best AFD vs d_intrinsic', fontweight='bold')
ax.set_xlabel('d_intrinsic'); ax.set_ylabel('Best AFD (ambient Fréchet distance)')
ax.legend(fontsize=7); ax.grid(alpha=0.2)

# ── (f) d* vs d_intrinsic — hypothesis test ──────────────────────────────────
ax = fig.add_subplot(gs[1, 2])
d_stars = []
for di in d_ints:
    best_afd_for_di = float('inf')
    best_dlat = d_lats[0]
    for c, dl in zip(cs, d_lats):
        h = H(c, di)
        afd_vals = [v for v in h.get('afd_test', []) if not np.isnan(v)]
        if afd_vals and min(afd_vals) < best_afd_for_di:
            best_afd_for_di = min(afd_vals)
            best_dlat = dl
    d_stars.append(best_dlat)
_lim = max(max(d_ints, default=1), max(d_stars, default=1)) * 1.15
ax.plot([0, _lim], [0, _lim], '--', color='#22c55e', lw=2, alpha=0.6,
        label='d* = d_int (ideal)')
ax.plot(d_ints, d_stars, 'o-', color='#2563eb', lw=2.5, ms=9, zorder=5,
        label='d* (min AFD)')
for di, ds in zip(d_ints, d_stars):
    ax.annotate(f'{ds}', (di, ds), textcoords='offset points',
                xytext=(6, 6), fontsize=8, fontweight='bold', color='#2563eb')
ax.set_xlim(0, _lim); ax.set_ylim(0, _lim)
ax.set_aspect('equal')
ax.set_title('(f) Hypothesis: d* ≈ d_intrinsic', fontweight='bold')
ax.set_xlabel('d_intrinsic (ground truth)'); ax.set_ylabel('d* (optimal d_latent)')
ax.legend(fontsize=8); ax.grid(alpha=0.2)

# ── (g) Diffusion loss curves ─────────────────────────────────────────────────
ax = fig.add_subplot(gs[2, 0])
for i, c in enumerate(cs):
    h = H(c)
    l_steps = h.get('loss_steps', [])
    l_vals  = h.get('loss_vals', [])
    if not l_vals: continue
    ax.plot(l_steps, l_vals, color=cmap_c[i], lw=1.5, alpha=0.85, label=f'c={c}')
ax.set_title(f'(g) Flow matching loss vs steps  (d_int={d_int_max})', fontweight='bold')
ax.set_xlabel('Training steps'); ax.set_ylabel('MSE loss')
ax.legend(fontsize=7.5, ncol=2); ax.grid(alpha=0.2)

# ── (h) FLIPD trajectory (multi-t) ────────────────────────────────────────────
ax = fig.add_subplot(gs[2, 1])
_flipd_ls = {t: ls for t, ls in zip(CFG['flipd_t_vals'],
                                      ['-', '--', '-.', ':'])}
for i, c in enumerate(cs):
    h = H(c)
    for t_val, ls in _flipd_ls.items():
        fdata = h.get('flipd', {}).get(t_val, {})
        fmean = np.array(fdata.get('mean', []))
        if not len(fmean): continue
        ax.plot(h['steps'], fmean, color=cmap_c[i], ls=ls, lw=1.4,
                label=f'c={c} t={t_val}')
ax.set_title(f'(h) FLIPD LID vs steps  (d_int={d_int_max})', fontweight='bold')
ax.set_xlabel('Training steps'); ax.set_ylabel('FLIPD LID estimate')
ax.legend(fontsize=6, ncol=3); ax.grid(alpha=0.2)

# ── (i) τ_gen and τ_mem vs d_intrinsic ────────────────────────────────────────
ax = fig.add_subplot(gs[2, 2])
for i, c in enumerate(cs):
    tg_by_di = [H(c, di).get('tau_gen') or CFG['total_steps'] for di in d_ints]
    tm_by_di = [H(c, di).get('tau_mem') or CFG['total_steps'] for di in d_ints]
    ax.plot(d_ints, tg_by_di, 'o-', color=cmap_c[i], lw=1.8, ms=5,
            label=f'd_lat={d_lats[i]}')
    ax.plot(d_ints, tm_by_di, 's--', color=cmap_c[i], lw=1.2, ms=4, alpha=0.6)
ax.plot([], [], 'ko-', lw=1.5, ms=4, label='τ_gen (solid)')
ax.plot([], [], 'ks--', lw=1, ms=3, alpha=0.6, label='τ_mem (dashed)')
ax.set_title('(i) τ_gen / τ_mem vs d_intrinsic', fontweight='bold')
ax.set_xlabel('d_intrinsic'); ax.set_ylabel('Training steps at onset')
ax.legend(fontsize=6, ncol=2); ax.grid(alpha=0.2)

# ── (j) Memorisation heatmap (d_latent × d_intrinsic) ─────────────────────────
ax = fig.add_subplot(gs[3, 0])
memo_grid = np.full((max(len(d_ints), 1), len(cs)), float('nan'))
for j, c in enumerate(cs):
    for i_d, di in enumerate(d_ints):
        h = all_histories.get((c, di))
        if h and h['memo_rate']:
            memo_grid[i_d, j] = h['memo_rate'][-1]
memo_max = np.nanmax(memo_grid) if memo_grid.size and not np.all(np.isnan(memo_grid)) else 0.2
im = ax.imshow(memo_grid, aspect='auto', origin='lower', cmap='RdYlGn_r',
               vmin=0, vmax=max(0.2, memo_max))
ax.set_xticks(range(len(cs))); ax.set_xticklabels(d_lats)
ax.set_yticks(range(len(d_ints))); ax.set_yticklabels(d_ints)
ax.set_xlabel('d_latent'); ax.set_ylabel('d_intrinsic')
ax.set_title('(j) Final memo rate  (d_latent × d_intrinsic)', fontweight='bold')
fig.colorbar(im, ax=ax, label='Memo rate', shrink=0.85)

# ── (k) FLIPD estimates vs d_intrinsic (ground truth = y=x) ───────────────────
ax = fig.add_subplot(gs[3, 1])
_pt = CFG['flipd_primary_t']
_lim_k = max(max(d_ints, default=1), max(d_lats, default=1)) * 1.1
ax.plot([0, _lim_k], [0, _lim_k], '-', color='#22c55e', lw=2.5, alpha=0.6,
        label='FLIPD = d_int (ideal)')
for i, c in enumerate(cs):
    flipd_by_di = []
    for di in d_ints:
        fdata = H(c, di).get('flipd', {}).get(_pt, {}).get('mean', [])
        flipd_by_di.append(fdata[-1] if fdata and not np.isnan(fdata[-1])
                           else float('nan'))
    ax.plot(d_ints, flipd_by_di, 'o-', color=cmap_c[i], lw=1.5, ms=5,
            label=f'd_lat={d_lats[i]}')
ax.set_title(f'(k) FLIPD (t={_pt}) vs d_intrinsic', fontweight='bold')
ax.set_xlabel('d_intrinsic (ground truth)'); ax.set_ylabel('FLIPD LID estimate')
ax.legend(fontsize=7); ax.grid(alpha=0.2)

# ── (l) Latent effective rank vs d_intrinsic ──────────────────────────────────
ax = fig.add_subplot(gs[3, 2])
for i, c in enumerate(cs):
    ranks_by_di = [eig_cache.get((c, di), {}).get('effective_rank', float('nan'))
                   for di in d_ints]
    ax.plot(d_ints, ranks_by_di, 'o-', color=cmap_c[i], lw=1.8, ms=5,
            label=f'd_lat={d_lats[i]}')
ax.plot(d_ints, d_ints, '--', color='#22c55e', lw=2, alpha=0.5,
        label='rank = d_int')
ax.set_title('(l) Latent effective rank vs d_intrinsic', fontweight='bold')
ax.set_xlabel('d_intrinsic'); ax.set_ylabel('Effective rank')
ax.legend(fontsize=7); ax.grid(alpha=0.2)

# ── (m) Generated effective rank vs steps ──────────────────────────────────────
ax = fig.add_subplot(gs[4, 0])
for i, c in enumerate(cs):
    h = H(c)
    ranks = h.get('gen_eff_rank', [])
    if not ranks: continue
    ax.plot(h['steps'], ranks, color=cmap_c[i], lw=1.8, label=f'c={c}')
    train_rank = eig_cache.get((c, d_int_max), {}).get('effective_rank')
    if train_rank is not None:
        ax.axhline(train_rank, color=cmap_c[i], ls=':', lw=1, alpha=0.5)
ax.set_title(f'(m) Generated effective rank vs steps  (d_int={d_int_max})', fontweight='bold')
ax.set_xlabel('Training steps'); ax.set_ylabel('Effective rank')
ax.legend(fontsize=7); ax.grid(alpha=0.2)

# ── (n) Final memo rate vs d_intrinsic ─────────────────────────────────────────
ax = fig.add_subplot(gs[4, 1])
for i, c in enumerate(cs):
    memo_by_di = []
    for di in d_ints:
        h = H(c, di)
        memo_by_di.append(h['memo_rate'][-1] if h['memo_rate'] else float('nan'))
    ax.plot(d_ints, memo_by_di, 'o-', color=cmap_c[i], lw=1.8, ms=5,
            label=f'd_lat={d_lats[i]}')
ax.axhline(CFG['memo_threshold'], color='#dc2626', ls='--', lw=1.5, alpha=0.7,
           label=f'τ_mem thresh ({CFG["memo_threshold"]})')
ax.set_title('(n) Final memo rate vs d_intrinsic', fontweight='bold')
ax.set_xlabel('d_intrinsic'); ax.set_ylabel('Memorisation rate')
ax.legend(fontsize=7); ax.grid(alpha=0.2)

# ── (o) AFD heatmap (d_latent × d_intrinsic) ──────────────────────────────────
ax = fig.add_subplot(gs[4, 2])
afd_grid = np.full((len(d_ints), len(cs)), float('nan'))
for j, c in enumerate(cs):
    for i_d, di in enumerate(d_ints):
        h = all_histories.get((c, di))
        if h and h.get('afd_test'):
            vals = [v for v in h['afd_test'] if not np.isnan(v)]
            if vals:
                afd_grid[i_d, j] = min(vals)
afd_max = np.nanmax(afd_grid) if not np.all(np.isnan(afd_grid)) else 100
im = ax.imshow(afd_grid, aspect='auto', origin='lower', cmap='RdYlGn_r',
               vmin=0, vmax=min(200, afd_max))
ax.set_xticks(range(len(cs))); ax.set_xticklabels(d_lats)
ax.set_yticks(range(len(d_ints))); ax.set_yticklabels(d_ints)
ax.set_xlabel('d_latent'); ax.set_ylabel('d_intrinsic')
ax.set_title('(o) Best AFD  (d_latent × d_intrinsic)', fontweight='bold')
fig.colorbar(im, ax=ax, label='Best AFD', shrink=0.85)

fig.suptitle(
    f'Hypersphere sweep — DiT-{CFG["dit_variant"]} | '
    f'd_int={d_ints}, d_ambient={CFG["d_ambient"]} | '
    f'flow matching | {CFG["total_steps"]:,} steps',
    fontsize=14, fontweight='bold', y=0.97
)

dash_path = os.path.join(OUT_DIR, 'synthetic_dashboard.png')
plt.savefig(dash_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Dashboard saved → {dash_path}')

---
## 10 · Styled summary table

In [ ]:
import pandas as pd
from IPython.display import display

rows = []
for (c, d_int), h in all_histories.items():
    d = c * CFG['latent_spatial']**2
    best_afd   = min(h.get('afd_test', [float('nan')])) if h.get('afd_test') else float('nan')
    best_lfd   = min(h.get('lfd_test', [float('nan')])) if h.get('lfd_test') else float('nan')
    final_memo = h['memo_rate'][-1] if h['memo_rate'] else float('nan')

    if   final_memo > CFG['memo_threshold']:      memo_flag = '🔴 memorising'
    elif final_memo > CFG['memo_threshold'] / 2:  memo_flag = '🟡 borderline'
    else:                                          memo_flag = '🟢 generalising'

    vae_bl = vae_baseline_cache.get((c, d_int), float('nan'))

    row = dict(
        c             = c,
        d_latent      = d,
        d_intrinsic   = d_int,
        VAE_baseline  = round(vae_bl, 2),
        best_AFD_test = round(best_afd, 2),
        best_LFD_test = round(best_lfd, 2),
        tau_gen       = h.get('tau_gen'),
        tau_mem       = h.get('tau_mem'),
        tau_gap       = (h['tau_mem'] - h['tau_gen']
                         if h.get('tau_mem') is not None and h.get('tau_gen') is not None
                         else None),
        final_memo    = round(final_memo, 3),
        memo_status   = memo_flag,
        gen_eff_rank  = round(h['gen_eff_rank'][-1], 1) if h.get('gen_eff_rank') else float('nan'),
    )
    for t_val in CFG['flipd_t_vals']:
        fdata = h.get('flipd', {}).get(t_val, {}).get('mean', [])
        row[f'FLIPD_t{t_val}'] = round(fdata[-1], 2) if fdata else float('nan')
    rows.append(row)

df = pd.DataFrame(rows)
csv_path = os.path.join(OUT_DIR, 'synthetic_results.csv')
df.to_csv(csv_path, index=False)
print(f'CSV saved → {csv_path}\n')

def col_memo(v):
    if v > CFG['memo_threshold']:       return 'background-color:#fee2e2;color:#991b1b'
    if v > CFG['memo_threshold'] / 2:   return 'background-color:#fef3c7;color:#92400e'
    return 'background-color:#dcfce7;color:#166534'

def col_gap(v):
    if v is None or (isinstance(v, float) and np.isnan(v)): return 'background-color:#f3f4f6;color:#6b7280'
    if v > 5000:  return 'background-color:#dcfce7;color:#166534'
    if v > 0:     return 'background-color:#fef3c7;color:#92400e'
    return 'background-color:#fee2e2;color:#991b1b'

def col_afd(v):
    if v < 30:  return 'background-color:#dcfce7;color:#166534'
    if v < 80:  return 'background-color:#fef3c7;color:#92400e'
    return 'background-color:#fee2e2;color:#991b1b'

styled = (
    df.style
      .map(col_memo, subset=['final_memo'])
      .map(col_gap,  subset=['tau_gap'])
      .map(col_afd,  subset=['best_AFD_test'])
      .format({**{'VAE_baseline': '{:.2f}', 'best_AFD_test': '{:.2f}',
                  'best_LFD_test': '{:.2f}', 'final_memo': '{:.3f}'},
               **{f'FLIPD_t{t}': '{:.2f}' for t in CFG['flipd_t_vals']}})
      .set_caption(
          f'Hypersphere sweep results — DiT-{CFG["dit_variant"]} | '
          f'd_int={d_ints}, d_ambient={CFG["d_ambient"]} '
          f'(flow matching, {CFG["total_steps"]:,} steps)')
      .set_table_styles([{'selector':'caption',
                          'props':[('font-size','13px'),('font-weight','bold'),
                                   ('padding-bottom','8px')]}])
)
display(styled)

# ── Final interpretation (per d_intrinsic) ────────────────────────────────────
d_ints = sorted(set(d for (_, d) in all_histories.keys()))
cs = CFG['latent_channels']
_pt = CFG['flipd_primary_t']

print('\n' + '='*65)
print('  HYPERSPHERE EXPERIMENT INTERPRETATION')
print('='*65)
print(f'  Ambient embedding     : R^{CFG["d_ambient"]}')
print(f'  Architecture          : DiT-{CFG["dit_variant"]}  (flow matching)')
print(f'  n_train               : {CFG["n_train"]:,}')
print(f'  d_intrinsic values    : {d_ints}')
print()

for d_int_val in d_ints:
    def _best_afd_for(c):
        h = all_histories.get((c, d_int_val), {})
        afds = [f for f in h.get('afd_test', [9999]) if not np.isnan(f)]
        if not afds:
            lfds = [f for f in h.get('lfd_test', [9999]) if not np.isnan(f)]
            return min(lfds) if lfds else 9999
        return min(afds)

    best_c = min(cs, key=_best_afd_for)
    best_d = best_c * CFG['latent_spatial']**2
    vae_bl_best = vae_baseline_cache.get((best_c, d_int_val), float('nan'))

    largest_c = max(cs)
    h_largest = all_histories.get((largest_c, d_int_val), {})
    _fdata = h_largest.get('flipd', {}).get(_pt, {}).get('mean', [])
    if _fdata and not np.isnan(_fdata[-1]):
        d_int_flipd = round(_fdata[-1], 2)
        flipd_src = f'c={largest_c}, t={_pt}'
    else:
        _all_f = []
        for c in cs:
            fd = all_histories.get((c, d_int_val), {}).get('flipd', {}).get(_pt, {}).get('mean', [])
            if fd and not np.isnan(fd[-1]):
                _all_f.append(fd[-1])
        d_int_flipd = round(max(_all_f), 2) if _all_f else 0
        flipd_src = f'max fallback, t={_pt}'

    sphere_n_val = d_int_val + 1
    print(f'  ── S^{d_int_val} ⊂ R^{sphere_n_val}  (d_intrinsic={d_int_val}) ──')
    print(f'    Best d_latent (d*)  : {best_d}  (c={best_c})')
    print(f'    VAE baseline AFD    : {vae_bl_best:.1f}')
    print(f'    FLIPD d_intrinsic   : {d_int_flipd}  ({flipd_src})')
    print(f'    |d* − d_int|/d_int  : '
          f'{abs(best_d - d_int_val) / (d_int_val + 1e-8) * 100:.1f}%')
    print()

print('='*65)

---
## 11 · Save artefacts to GCS  *(skip on Kaggle)*

In [ ]:
GCS_BUCKET = os.environ.get('GCS_BUCKET', '')

if IN_GCP and GCS_BUCKET:
    import re
    from google.cloud import storage
    match = re.match(r'gs://([^/]+)/?(.*)', GCS_BUCKET)
    bucket_name, prefix = match.group(1), match.group(2)
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    for fname in os.listdir(OUT_DIR):
        if fname.endswith(('.png', '.csv', '.pt')):
            blob_path = f'{prefix}/{fname}' if prefix else fname
            bucket.blob(blob_path).upload_from_filename(
                os.path.join(OUT_DIR, fname))
            print(f'Uploaded: gs://{bucket_name}/{blob_path}')
elif IN_KAGGLE:
    print('Kaggle: artefacts in /kaggle/working/synthetic/ (visible in output tab).')
else:
    print(f'Local: artefacts in {OUT_DIR}\nSet GCS_BUCKET env var to upload.')